In [4]:
import numpy as np
import os
import functions as f
from scipy import ndimage
from rasterio.features import rasterize
from rasterio.transform import from_origin
import xarray as xr


/home/angelo/.local/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/home/angelo/.local/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [ ]:
def save_time_series(
    path_dataset,
    option,
    date_start,
    date_end,
    year,
    crops,
    border_erosion_size,
    baseline=6,
    output_dir=None,
    show_series=True,
    save_series=False,
    save_figures=False
):

    # ============================================================
    # 1. ABRIR DATASET
    # ============================================================

    ds = xr.open_dataset(path_dataset)

    try:

        # ========================================================
        # 2. COMPROBAR OPCIÓN
        # ========================================================

        option = option.lower()

        valid_options = [
            "backscatter",
            "backscatter_coherencia",
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]

        if option not in valid_options:
            raise ValueError(
                "option debe ser una de: "
                f"{valid_options}"
            )
        if show_series:
            raise NotImplementedError("show_series aún no está implementado")

        if save_figures:
            raise NotImplementedError("save_figures aún no está implementado")
        # ========================================================
        # 3. COMPROBAR AÑO
        # ========================================================

        years_dataset = np.asarray(
            ds["years"].values,
            dtype=int
        )

        if year not in years_dataset:
            raise ValueError(
                f"Año inválido: {year}. "
                f"Años disponibles: "
                f"{years_dataset.tolist()}"
            )

        year_idx = np.where(
            years_dataset == year
        )[0][0]

        # ========================================================
        # 4. CULTIVOS
        # ========================================================

        crop_types = [
            str(c) for c in ds["crops"].values
        ]

        crop_types_lower = [
            c.lower() for c in crop_types
        ]

        if isinstance(crops, str):
            crops = [crops]

        for crop in crops:

            if crop.lower() not in crop_types_lower:
                raise ValueError(
                    f"Cultivo no válido: {crop}"
                )

        # ========================================================
        # 5. MAPA DE CULTIVOS
        # ========================================================

        crop_map = ds[
            "crop_type_map"
        ].isel(
            year=year_idx
        ).values

        # ========================================================
        # 6. EROSIÓN
        # ========================================================

        if border_erosion_size > 1:

            structure = np.ones(
                (
                    border_erosion_size,
                    border_erosion_size
                ),
                dtype=bool
            )

            crop_map_eroded = np.zeros_like(
                crop_map
            )

            for crop_code in np.unique(
                crop_map
            ):

                if crop_code == 0:
                    continue

                mask_crop = (
                    crop_map == crop_code
                )

                eroded_mask = ndimage.binary_erosion(
                    mask_crop,
                    structure=structure
                )

                crop_map_eroded[
                    eroded_mask
                ] = crop_code

        else:

            crop_map_eroded = crop_map.copy()

        # ========================================================
        # 7. VARIABLES
        # ========================================================

        VV_all = None
        VH_all = None

        COH_VV_all = None
        COH_VH_all = None

        NDVI_all = None
        NIR_all = None

        dates = None
        dates_optical = None

        # ========================================================
        # 8. BACKSCATTER
        # ========================================================

        if option == "backscatter":

            dates_all = np.asarray(
                ds["times_C2"].values,
                dtype="datetime64[D]"
            )

            mask_dates = (
                (dates_all >= np.datetime64(date_start))
                &
                (dates_all <= np.datetime64(date_end))
            )

            idx_dates = np.where(
                mask_dates
            )[0]

            if len(idx_dates) == 0:

                raise ValueError(
                    "No hay fechas SAR dentro "
                    "del intervalo seleccionado."
                )

            dates = dates_all[idx_dates]

            print("\n" + "=" * 60)
            print("BACKSCATTER")
            print("=" * 60)

            print(f"Fechas SAR: {len(dates)}")
            print(f"Desde: {dates[0]}")
            print(f"Hasta: {dates[-1]}")

            s1 = ds["S1_C2"].isel(
                time_C2=idx_dates
            )

            # C11 = VV
            # C22 = VH

            VV_linear = s1.isel(
                element_C2=0
            ).values

            VH_linear = s1.isel(
                element_C2=3
            ).values

            eps = 1e-10

            VV_all = 10 * np.log10(
                np.maximum(
                    VV_linear,
                    eps
                )
            )

            VH_all = 10 * np.log10(
                np.maximum(
                    VH_linear,
                    eps
                )
            )

        # ========================================================
        # 9. BACKSCATTER + COHERENCIA
        # ========================================================

        elif option in [
            "backscatter_coherencia",
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            if baseline not in [6, 12]:

                raise ValueError(
                    "baseline debe ser 6 o 12."
                )

            # ====================================================
            # FECHAS BACKSCATTER
            # ====================================================

            dates_c2_all = np.asarray(
                ds["times_C2"].values,
                dtype="datetime64[D]"
            )

            mask_c2 = (
                (dates_c2_all >= np.datetime64(date_start))
                &
                (dates_c2_all <= np.datetime64(date_end))
            )

            idx_c2_periodo = np.where(
                mask_c2
            )[0]

            dates_c2 = dates_c2_all[
                idx_c2_periodo
            ]

            if len(dates_c2) == 0:

                raise ValueError(
                    "No hay fechas de backscatter "
                    "dentro del intervalo."
                )

            # ====================================================
            # FECHAS COHERENCIA
            # ====================================================

            time_dim = (
                f"time_coher_{baseline:02d}days"
            )

            time_var = (
                f"times_coher_{baseline:02d}days"
            )

            dates_coh_all = np.asarray(
                ds[time_var].values,
                dtype="datetime64[D]"
            )

            mask_coh = (
                (dates_coh_all >= np.datetime64(date_start))
                &
                (dates_coh_all <= np.datetime64(date_end))
            )

            idx_coh_periodo = np.where(
                mask_coh
            )[0]

            dates_coh = dates_coh_all[
                idx_coh_periodo
            ]

            if len(dates_coh) == 0:

                raise ValueError(
                    "No hay fechas de coherencia "
                    "dentro del intervalo."
                )

            # ====================================================
            # FECHAS COMUNES
            # ====================================================

            dates_comunes = np.intersect1d(
                dates_c2,
                dates_coh
            )

            if len(dates_comunes) == 0:

                raise ValueError(
                    "No existen fechas comunes entre "
                    "backscatter y coherencia."
                )

            # Índices originales

            idx_c2 = np.where(
                np.isin(
                    dates_c2_all,
                    dates_comunes
                )
            )[0]

            idx_coh = np.where(
                np.isin(
                    dates_coh_all,
                    dates_comunes
                )
            )[0]

            dates = dates_comunes

            print("\n" + "=" * 60)
            print("BACKSCATTER + COHERENCIA")
            print("=" * 60)

            print(
                f"Baseline: {baseline} días"
            )

            print(
                f"Fechas comunes: {len(dates)}"
            )

            print(
                f"Desde: {dates[0]}"
            )

            print(
                f"Hasta: {dates[-1]}"
            )

            # ====================================================
            # BACKSCATTER
            # ====================================================

            s1 = ds["S1_C2"].isel(
                time_C2=idx_c2
            )

            VV_linear = s1.isel(
                element_C2=0
            ).values

            VH_linear = s1.isel(
                element_C2=3
            ).values

            eps = 1e-10

            VV_all = 10 * np.log10(
                np.maximum(
                    VV_linear,
                    eps
                )
            )

            VH_all = 10 * np.log10(
                np.maximum(
                    VH_linear,
                    eps
                )
            )

            # ====================================================
            # COHERENCIA
            # ====================================================

            var_coh = (
                f"S1_{baseline:02d}days_coherence"
            )

            coh = ds[var_coh].isel(
                **{
                    time_dim: idx_coh
                }
            )

            COH_VV_all = coh.isel(
                channel_Cband=0
            ).values

            COH_VH_all = coh.isel(
                channel_Cband=1
            ).values

            print(
                f"VV shape: {VV_all.shape}"
            )

            print(
                f"VH shape: {VH_all.shape}"
            )

            print(
                f"COH VV shape: "
                f"{COH_VV_all.shape}"
            )

            print(
                f"COH VH shape: "
                f"{COH_VH_all.shape}"
            )

        # ========================================================
        # 10. DATOS ÓPTICOS
        # ========================================================

        if option in [
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            dates_optical_all = np.asarray(
                ds["times_optical"].values,
                dtype="datetime64[D]"
            )

            mask_optical = (
                (dates_optical_all >= np.datetime64(date_start))
                &
                (dates_optical_all <= np.datetime64(date_end))
            )

            idx_optical = np.where(
                mask_optical
            )[0]

            if len(idx_optical) == 0:

                raise ValueError(
                    "No hay fechas ópticas dentro "
                    "del intervalo seleccionado."
                )

            dates_optical = (
                dates_optical_all[idx_optical]
            )

            print("\n" + "=" * 60)
            print("DATOS ÓPTICOS")
            print("=" * 60)

            print(
                f"Fechas ópticas: "
                f"{len(dates_optical)}"
            )

            print(
                f"Desde: "
                f"{dates_optical[0]}"
            )

            print(
                f"Hasta: "
                f"{dates_optical[-1]}"
            )

            # ====================================================
            # NDVI
            # ====================================================

            if option in [
                "backscatter_coherencia_ndvi",
                "backscatter_coherencia_ndvi_nir"
            ]:

                NDVI_all = ds["S2"].isel(
                    time_optical=idx_optical,
                    band_optical=4
                ).values

                print(
                    f"NDVI shape: "
                    f"{NDVI_all.shape}"
                )

            # ====================================================
            # NIR
            # ====================================================

            if option in [
                "backscatter_coherencia_nir",
                "backscatter_coherencia_ndvi_nir"
            ]:

                NIR_all = ds["S2"].isel(
                    time_optical=idx_optical,
                    band_optical=3
                ).values

                print(
                    f"NIR shape: "
                    f"{NIR_all.shape}"
                )

        # ========================================================
        # 11. LISTAS
        # ========================================================

        VV_list = []
        VH_list = []

        COH_VV_list = []
        COH_VH_list = []

        NDVI_list = []
        NIR_list = []

        coords_list = []
        crop_names_list = []

        # ========================================================
        # 12. EXTRAER PÍXELES
        # ========================================================

        print("\n" + "=" * 60)
        print("EXTRACCIÓN DE CULTIVOS")
        print("=" * 60)

        for crop in crops:

            crop_code = (
                crop_types_lower.index(
                    crop.lower()
                ) + 1
            )

            mask = (
                crop_map_eroded == crop_code
            )

            y_coords, x_coords = np.where(
                mask
            )

            n_pixels = len(y_coords)

            if n_pixels == 0:

                print(
                    f"⚠ {crop}: sin píxeles"
                )

                continue

            print(
                f"{crop}: "
                f"{n_pixels:,} píxeles"
            )
            
            # ====================================================
            # FILTRO: píxeles con ceros en todas las fechas
            # ====================================================

            VV_pix_check = VV_all[
                :,
                y_coords,
                x_coords
            ].T  # shape: (n_pixeles, 61)

            validos = ~(VV_pix_check < -99.9).all(axis=1)
            y_coords = y_coords[validos]
            x_coords = x_coords[validos]

            print(
                f"  → {(~validos).sum()} píxeles "
                f"excluidos por ceros en todas las fechas"
            )

            # ====================================================
            # VV / VH
            # ====================================================

            VV_pix = VV_all[
                :,
                y_coords,
                x_coords
            ].T

            VH_pix = VH_all[
                :,
                y_coords,
                x_coords
            ].T

            VV_list.append(
                VV_pix
            )

            VH_list.append(
                VH_pix
            )

            # ====================================================
            # COHERENCIA
            # ====================================================

            if option in [
                "backscatter_coherencia",
                "backscatter_coherencia_ndvi",
                "backscatter_coherencia_nir",
                "backscatter_coherencia_ndvi_nir"
            ]:

                COH_VV_pix = COH_VV_all[
                    :,
                    y_coords,
                    x_coords
                ].T

                COH_VH_pix = COH_VH_all[
                    :,
                    y_coords,
                    x_coords
                ].T

                COH_VV_list.append(
                    COH_VV_pix
                )

                COH_VH_list.append(
                    COH_VH_pix
                )

            # ====================================================
            # NDVI
            # ====================================================

            if option in [
                "backscatter_coherencia_ndvi",
                "backscatter_coherencia_ndvi_nir"
            ]:

                NDVI_pix = NDVI_all[
                    :,
                    y_coords,
                    x_coords
                ].T

                NDVI_list.append(
                    NDVI_pix
                )

            # ====================================================
            # NIR
            # ====================================================

            if option in [
                "backscatter_coherencia_nir",
                "backscatter_coherencia_ndvi_nir"
            ]:

                NIR_pix = NIR_all[
                    :,
                    y_coords,
                    x_coords
                ].T

                NIR_list.append(
                    NIR_pix
                )

            # ====================================================
            # COORDENADAS
            # ====================================================

            coords_crop = np.column_stack(
                (y_coords, x_coords)
            )

            coords_list.append(
                coords_crop
            )

            crop_names_list.append(
                crop
            )

        # ========================================================
        # 13. COMPROBAR RESULTADOS
        # ========================================================

        if not VV_list:

            raise ValueError(
                "No se encontraron píxeles "
                "para los cultivos seleccionados."
            )

        # ========================================================
        # 14. NORMALIZACIÓN VV / VH
        # ========================================================

        VV_raw = np.concatenate(
            VV_list,
            axis=0
        )

        VH_raw = np.concatenate(
            VH_list,
            axis=0
        )

        VV_min = np.nanpercentile(
            VV_raw,
            1
        )

        VV_max = np.nanpercentile(
            VV_raw,
            99
        )

        VH_min = np.nanpercentile(
            VH_raw,
            1
        )

        VH_max = np.nanpercentile(
            VH_raw,
            99
        )

        print("\n" + "=" * 60)
        print("NORMALIZACIÓN")
        print("=" * 60)

        print(
            f"VV → P1={VV_min:.4f} | "
            f"P99={VV_max:.4f}"
        )

        print(
            f"VH → P1={VH_min:.4f} | "
            f"P99={VH_max:.4f}"
        )

        denom_vv = (
            VV_max - VV_min
        )

        denom_vh = (
            VH_max - VH_min
        )

        # ========================================================
        # 15. NORMALIZACIÓN NIR
        # ========================================================

        NIR_min = None
        NIR_max = None
        denom_nir = None

        if option in [
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            NIR_raw = np.concatenate(
                NIR_list,
                axis=0
            )

            NIR_min = np.nanpercentile(
                NIR_raw,
                1
            )

            NIR_max = np.nanpercentile(
                NIR_raw,
                99
            )

            denom_nir = (
                NIR_max - NIR_min
            )

            print(
                f"NIR → P1={NIR_min:.4f} | "
                f"P99={NIR_max:.4f}"
            )

        # ========================================================
        # 16. LISTAS NORMALIZADAS
        # ========================================================
        # NOTA: los percentiles se calculan sobre todos los cultivos
        # pasados en la llamada. Si se llama con un subconjunto,
        # la normalización cambia y los .npz no son comparables.
        # Llamar siempre con los 19 cultivos juntos.
        
        VV_list_norm = []
        VH_list_norm = []

        COH_VV_list_norm = []
        COH_VH_list_norm = []

        NDVI_list_norm = []
        NIR_list_norm = []

        for i in range(
            len(VV_list)
        ):

            # ====================================================
            # VV
            # ====================================================

            VV_pix = VV_list[i]

            if denom_vv > 0:

                VV_norm = (
                    VV_pix - VV_min
                ) / denom_vv

            else:

                VV_norm = np.zeros_like(
                    VV_pix
                )

            VV_norm = np.clip(
                VV_norm,
                0.0,
                1.0
            )

            VV_norm = np.nan_to_num(
                VV_norm,
                nan=0.0,
                posinf=1.0,
                neginf=0.0
            )

            VV_list_norm.append(
                VV_norm.astype(
                    np.float32
                )
            )

            # ====================================================
            # VH
            # ====================================================

            VH_pix = VH_list[i]

            if denom_vh > 0:

                VH_norm = (
                    VH_pix - VH_min
                ) / denom_vh

            else:

                VH_norm = np.zeros_like(
                    VH_pix
                )

            VH_norm = np.clip(
                VH_norm,
                0.0,
                1.0
            )

            VH_norm = np.nan_to_num(
                VH_norm,
                nan=0.0,
                posinf=1.0,
                neginf=0.0
            )

            VH_list_norm.append(
                VH_norm.astype(
                    np.float32
                )
            )

            # ====================================================
            # COHERENCIA
            # ====================================================

            if option in [
                "backscatter_coherencia",
                "backscatter_coherencia_ndvi",
                "backscatter_coherencia_nir",
                "backscatter_coherencia_ndvi_nir"
            ]:

                COH_VV_pix = (
                    COH_VV_list[i]
                )

                COH_VH_pix = (
                    COH_VH_list[i]
                )

                COH_VV_norm = np.nan_to_num(
                    COH_VV_pix,
                    nan=0.0,
                    posinf=1.0,
                    neginf=0.0
                )

                COH_VH_norm = np.nan_to_num(
                    COH_VH_pix,
                    nan=0.0,
                    posinf=1.0,
                    neginf=0.0
                )

                COH_VV_norm = np.clip(
                    COH_VV_norm,
                    0.0,
                    1.0
                )

                COH_VH_norm = np.clip(
                    COH_VH_norm,
                    0.0,
                    1.0
                )

                COH_VV_list_norm.append(
                    COH_VV_norm.astype(
                        np.float32
                    )
                )

                COH_VH_list_norm.append(
                    COH_VH_norm.astype(
                        np.float32
                    )
                )

            # ====================================================
            # NDVI
            # ====================================================

            if option in [
                "backscatter_coherencia_ndvi",
                "backscatter_coherencia_ndvi_nir"
            ]:

                NDVI_pix = NDVI_list[i]

                NDVI_norm = np.nan_to_num(
                    NDVI_pix,
                    nan=0.0,
                    posinf=1.0,
                    neginf=-1.0
                )

                NDVI_norm = np.clip(
                    NDVI_norm,
                    -1.0,
                    1.0
                )

                NDVI_list_norm.append(
                    NDVI_norm.astype(
                        np.float32
                    )
                )

            # ====================================================
            # NIR
            # ====================================================

            if option in [
                "backscatter_coherencia_nir",
                "backscatter_coherencia_ndvi_nir"
            ]:

                NIR_pix = NIR_list[i]

                if denom_nir > 0:

                    NIR_norm = (
                        NIR_pix - NIR_min
                    ) / denom_nir

                else:

                    NIR_norm = np.zeros_like(
                        NIR_pix
                    )

                NIR_norm = np.clip(
                    NIR_norm,
                    0.0,
                    1.0
                )

                NIR_norm = np.nan_to_num(
                    NIR_norm,
                    nan=0.0,
                    posinf=1.0,
                    neginf=0.0
                )

                NIR_list_norm.append(
                    NIR_norm.astype(
                        np.float32
                    )
                )

        # ========================================================
        # 17. CREAR DIRECTORIO
        # ========================================================

        if (
            output_dir
            and (
                save_series
                or save_figures
            )
        ):

            os.makedirs(
                output_dir,
                exist_ok=True
            )

        # ========================================================
        # 18. GUARDAR POR CULTIVO
        # ========================================================

        for i, crop in enumerate(
            crop_names_list
        ):

            VV_pix = VV_list_norm[i]
            VH_pix = VH_list_norm[i]

            coords_crop = coords_list[i]

            if save_series:

                if output_dir is None:

                    raise ValueError(
                        "output_dir es necesario "
                        "para guardar."
                    )

                datos_guardar = {

                    "VV": VV_pix,
                    "VH": VH_pix,

                    "dates": dates,
                    "coords": coords_crop,

                    "year": year,
                    "option": option,

                    "border_erosion_size":
                        border_erosion_size
                }

                # =================================================
                # COHERENCIA
                # =================================================

                if option in [
                    "backscatter_coherencia",
                    "backscatter_coherencia_ndvi",
                    "backscatter_coherencia_nir",
                    "backscatter_coherencia_ndvi_nir"
                ]:

                    datos_guardar.update({

                        "COH_VV":
                            COH_VV_list_norm[i],

                        "COH_VH":
                            COH_VH_list_norm[i],

                        "coherence_baseline":
                            baseline
                    })

                # =================================================
                # NDVI
                # =================================================

                if option in [
                    "backscatter_coherencia_ndvi",
                    "backscatter_coherencia_ndvi_nir"
                ]:

                    datos_guardar.update({

                        "NDVI":
                            NDVI_list_norm[i],

                        "dates_optical":
                            dates_optical
                    })

                # =================================================
                # NIR
                # =================================================

                if option in [
                    "backscatter_coherencia_nir",
                    "backscatter_coherencia_ndvi_nir"
                ]:

                    datos_guardar.update({

                        "NIR":
                            NIR_list_norm[i],

                        "dates_optical":
                            dates_optical
                    })

                # =================================================
                # GUARDAR
                # =================================================

                np.savez_compressed(
                    

                    os.path.join(
                        output_dir,
                        f"{crop}_series_{year}.npz"
                    ),

                    **datos_guardar
                )

        # ========================================================
        # 19. CONCATENAR RESULTADOS
        # ========================================================

        VV_final = np.concatenate(
            VV_list_norm,
            axis=0
        )

        VH_final = np.concatenate(
            VH_list_norm,
            axis=0
        )

        coords_final = np.concatenate(
            coords_list,
            axis=0
        )

        # ========================================================
        # COHERENCIA
        # ========================================================

        COH_VV_final = None
        COH_VH_final = None

        if option in [
            "backscatter_coherencia",
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            COH_VV_final = np.concatenate(
                COH_VV_list_norm,
                axis=0
            )

            COH_VH_final = np.concatenate(
                COH_VH_list_norm,
                axis=0
            )

        # ========================================================
        # NDVI
        # ========================================================

        NDVI_final = None

        if option in [
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_ndvi_nir"
        ]:

            NDVI_final = np.concatenate(
                NDVI_list_norm,
                axis=0
            )

        # ========================================================
        # NIR
        # ========================================================

        NIR_final = None

        if option in [
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            NIR_final = np.concatenate(
                NIR_list_norm,
                axis=0
            )

        # ========================================================
        # 20. INFORMACIÓN FINAL
        # ========================================================

        print("\n" + "=" * 60)
        print("RESULTADO FINAL")
        print("=" * 60)

        print(
            f"Opción: {option}"
        )

        print(
            f"Píxeles: "
            f"{len(coords_final):,}"
        )

        print(
            f"VV shape: "
            f"{VV_final.shape}"
        )

        print(
            f"VH shape: "
            f"{VH_final.shape}"
        )

        if COH_VV_final is not None:

            print(
                f"COH VV shape: "
                f"{COH_VV_final.shape}"
            )

            print(
                f"COH VH shape: "
                f"{COH_VH_final.shape}"
            )

        if NDVI_final is not None:

            print(
                f"NDVI shape: "
                f"{NDVI_final.shape}"
            )

        if NIR_final is not None:

            print(
                f"NIR shape: "
                f"{NIR_final.shape}"
            )

        if dates_optical is not None:

            print(
                f"Fechas ópticas: "
                f"{len(dates_optical)}"
            )

        # ========================================================
        # 21. RETURN
        # ========================================================

        return_data = [
            coords_final,
            VV_final,
            VH_final
        ]

        # --------------------------------------------------------
        # COHERENCIA
        # --------------------------------------------------------

        if option in [
            "backscatter_coherencia",
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            return_data.extend([
                COH_VV_final,
                COH_VH_final
            ])

        # --------------------------------------------------------
        # NDVI
        # --------------------------------------------------------

        if option in [
            "backscatter_coherencia_ndvi",
            "backscatter_coherencia_ndvi_nir"
        ]:

            return_data.append(
                NDVI_final
            )

        # --------------------------------------------------------
        # NIR
        # --------------------------------------------------------

        if option in [
            "backscatter_coherencia_nir",
            "backscatter_coherencia_ndvi_nir"
        ]:

            return_data.append(
                NIR_final
            )

        # --------------------------------------------------------
        # FECHAS SAR
        # --------------------------------------------------------

        return_data.append(
            dates
        )

        # --------------------------------------------------------
        # FECHAS ÓPTICAS
        # --------------------------------------------------------

        if dates_optical is not None:

            return_data.append(
                dates_optical
            )

        return tuple(return_data)

    finally:

        ds.close()

In [13]:
path_dataset = r"/temusa_store/Processed_Data/Sentinel1/Sevilla/SLC/Dataset_2017-21.nc"   # Ruta local del dataset grande
output_dir = "/home/angelo/MyStore/Proyecto Investigador/2017/series_temporales_2017_normalized(backscatter)/"  # Directorio de salida para guardar los resultados
cultivos = [
    'Alfalfa',
    'Barley',
    'Carrot',
    'Chickpea',
    'Cotton',
    'Fallow',
    'Maize',
    'Oats',
    'Onion',
    'Pepper',
    'Potato',
    'Pumpkin',
    'Quinoa',
    'Rice',
    'SugarBeet',
    'Sunflower',
    'SweetPotato',
    'Tomato',
    'Wheat'
]

resultado = save_time_series(

    path_dataset=path_dataset,
    option="backscatter",
    date_start="2017-01-01",
    date_end="2017-12-31",
    year=2017,
    crops=cultivos,
    baseline=6,
    border_erosion_size=3,
    output_dir=output_dir,
    show_series=False,
    save_series=True,
    save_figures=False
)


BACKSCATTER
Fechas SAR: 61
Desde: 2017-01-03
Hasta: 2017-12-29

EXTRACCIÓN DE CULTIVOS
Alfalfa: 31,725 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
⚠ Barley: sin píxeles
Carrot: 15,835 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Chickpea: 3,487 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Cotton: 476,886 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Fallow: 5,543 píxeles
  → 25 píxeles excluidos por ceros en todas las fechas
Maize: 50,749 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
⚠ Oats: sin píxeles
Onion: 10,838 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Pepper: 4,261 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Potato: 2,716 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Pumpkin: 7,172 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Quinoa: 7,920 píxeles
  → 0 píxeles excluidos por ceros en todas las fechas
Rice: 230,055 píxeles
  →

In [12]:
coords, VV, VH, dates = resultado
arrays = {
    "VV": VV,
    "VH": VH,
    
}

for nombre, arr in arrays.items():
    print(f"{nombre}: shape={arr.shape} | min={np.nanmin(arr):.4f} | max={np.nanmax(arr):.4f} | mean={np.nanmean(arr):.4f}")

VV: shape=(1367992, 61) | min=0.0000 | max=1.0000 | mean=0.6141
VH: shape=(1367992, 61) | min=0.0000 | max=1.0000 | mean=0.5404
